# MGT-353 Supply Chain Analytics — Portfolio Milestone 1

## Honda Atlas Cars (Pakistan) Limited — Honda City 1.2L

**Student:** Syed Uzair Ali  
**IBA ERP ID:** 24763  
**Course:** MGT-353 Supply Chain Analytics  
**Instructor:** Mr. Faisal Jalal

> **Company-Calibrated Synthetic Dataset for Academic Analysis — illustrative figures, calibrated to Honda Atlas Cars (Pakistan) Limited's disclosed scale, not actual company records.**

This notebook reproduces the forecasting, validation, regression, error analysis, and stress-testing workflow used in the submission package.


## 1. Setup and data loading

Upload the frozen CSV when running this notebook in Google Colab. The notebook deliberately reads the frozen dataset rather than regenerating demand, so model comparison remains auditable.


In [ ]:
# Colab data upload
from google.colab import files
uploaded = files.upload()

import io, os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")

csv_files = [f for f in uploaded.keys() if f.lower().endswith(".csv")]
assert csv_files, "Please upload the synthetic dataset CSV."
df = pd.read_csv(csv_files[0], parse_dates=["Week_Start_Date"])
df = df.sort_values("Week_Start_Date").reset_index(drop=True)

print("Shape:", df.shape)
display(df.head())
display(df.tail())


## 2. Data quality and descriptive statistics


In [ ]:
required_cols = [
    "Week_Start_Date","Product_or_Category","Actual_Demand","Historical_Forecast",
    "Price_Index","Promo_Flag","Promo_Depth_Pct","Event_Flag","Economic_Index",
    "Stockout_Flag","Exceptional_Event","Event_Description"
]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"

print("Date range:", df["Week_Start_Date"].min().date(), "to", df["Week_Start_Date"].max().date())
print("Duplicate dates:", df["Week_Start_Date"].duplicated().sum())
print("Missing demand:", df["Actual_Demand"].isna().sum())

stats = df["Actual_Demand"].agg(["mean","median","std","min","max"])
stats["range"] = stats["max"] - stats["min"]
stats["CV_%"] = stats["std"] / stats["mean"] * 100
display(stats.to_frame("Value"))

promo_summary = df.groupby("Promo_Flag")["Actual_Demand"].mean()
print("Promotion-week mean:", round(promo_summary.get(1, np.nan),2))
print("Non-promotion mean:", round(promo_summary.get(0, np.nan),2))


## 3. Demand pattern

The series is interpreted as level-dominated with mild drift, event peaks, promotion uplift, and isolated stockout suppression. These features support responsive smoothing while still requiring event/availability monitoring.


In [ ]:
plt.figure(figsize=(14,5))
plt.plot(df["Week_Start_Date"], df["Actual_Demand"], label="Actual demand")
events = df[df["Event_Flag"] == 1]
plt.scatter(events["Week_Start_Date"], events["Actual_Demand"], label="Event weeks")
plt.xlabel("Week")
plt.ylabel("Units")
plt.title("Honda City 1.2L Synthetic Weekly Demand")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 4. Train / holdout design

The final 12 weeks are the frozen holdout. Candidate time-series models are fitted only on the first 144 observations. Holdout actuals are used only after forecasts are produced.


In [ ]:
train_df = df.iloc[:-12].copy()
test_df = df.iloc[-12:].copy()

print("Train:", train_df["Week_Start_Date"].min().date(), "to", train_df["Week_Start_Date"].max().date(), len(train_df))
print("Holdout:", test_df["Week_Start_Date"].min().date(), "to", test_df["Week_Start_Date"].max().date(), len(test_df))
assert len(test_df) == 12


## 5. Forecasting models

Models compared on the identical holdout:
1. Naive
2. Seasonal Naive (52)
3. 4-week Moving Average
4. SES α=0.3
5. SES α=0.6
6. Holt
7. Holt-Winters additive (52)

All models use the same target and the same final 12-week test window.


In [ ]:
y_train = train_df["Actual_Demand"].astype(float)
h = len(test_df)

forecasts = {}

# Naive
forecasts["Naive"] = np.repeat(y_train.iloc[-1], h)

# Seasonal naive, lag 52
seasonal_lag = 52
forecasts["Seasonal Naive (52)"] = np.array([y_train.iloc[-seasonal_lag + i % seasonal_lag] for i in range(h)])

# 4-week moving average
ma4 = y_train.tail(4).mean()
forecasts["4-Week Moving Average"] = np.repeat(ma4, h)

# SES
ses03 = SimpleExpSmoothing(y_train, initialization_method="estimated").fit(smoothing_level=0.3, optimized=False)
ses06 = SimpleExpSmoothing(y_train, initialization_method="estimated").fit(smoothing_level=0.6, optimized=False)
forecasts["SES α=0.3"] = ses03.forecast(h).to_numpy()
forecasts["SES α=0.6"] = ses06.forecast(h).to_numpy()

# Holt
holt = Holt(y_train, initialization_method="estimated").fit(optimized=True)
forecasts["Holt"] = holt.forecast(h).to_numpy()

# Holt-Winters additive seasonality
hw = ExponentialSmoothing(
    y_train, trend="add", seasonal="add", seasonal_periods=52,
    initialization_method="estimated"
).fit(optimized=True)
forecasts["Holt-Winters Additive (52)"] = hw.forecast(h).to_numpy()

# Holdout plot
plt.figure(figsize=(14,5))
plt.plot(test_df["Week_Start_Date"], test_df["Actual_Demand"], marker="o", linewidth=2, label="Actual")
for name, fc in forecasts.items():
    plt.plot(test_df["Week_Start_Date"], fc, marker=".", alpha=0.55, label=name)
plt.title("Common 12-Week Holdout: Actual vs Candidate Forecasts")
plt.xlabel("Week")
plt.ylabel("Units")
plt.legend(ncol=2, fontsize=8)
plt.grid(alpha=0.25)
plt.show()


## 6. Forecast accuracy metrics

**Error = Actual − Forecast** throughout. This convention determines the sign interpretation of Bias, RSFE and Tracking Signal.


In [ ]:
def metric_row(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    err = actual - forecast
    abs_err = np.abs(err)
    mae = abs_err.mean()
    mape = (abs_err / np.where(actual == 0, np.nan, np.abs(actual))).mean() * 100
    rmse = np.sqrt(np.mean(err**2))
    wape = abs_err.sum() / np.abs(actual).sum() * 100
    bias = err.mean()
    rsfe = err.sum()
    ts = rsfe / mae if mae != 0 else np.nan
    return {
        "MAD/MAE": mae, "MAPE": mape, "RMSE": rmse, "WAPE": wape,
        "Bias/MFE": bias, "RSFE": rsfe, "Tracking Signal": ts
    }

comparison = []
for name, fc in forecasts.items():
    row = {"Model": name}
    row.update(metric_row(test_df["Actual_Demand"], fc))
    comparison.append(row)

comparison_df = pd.DataFrame(comparison).sort_values(["MAD/MAE","RMSE"]).reset_index(drop=True)
display(comparison_df.round(2))


## 7. Model selection

The selected model is SES α=0.6 because it performs strongly across multiple primary metrics while maintaining a manageable bias profile. Selection is not based on MAPE alone.


In [ ]:
selected_model = "SES α=0.6"
selected_fc = forecasts[selected_model]
selected_metrics = metric_row(test_df["Actual_Demand"], selected_fc)
print(selected_model)
for k,v in selected_metrics.items():
    print(f"{k}: {v:.2f}")


## 8. Holdout error diagnostics and five largest misses


In [ ]:
holdout = test_df[["Week_Start_Date","Actual_Demand","Promo_Flag","Event_Flag","Stockout_Flag"]].copy()
holdout["Forecast"] = selected_fc
holdout["Error"] = holdout["Actual_Demand"] - holdout["Forecast"]
holdout["Abs_Error"] = holdout["Error"].abs()
top5 = holdout.sort_values("Abs_Error", ascending=False).head(5)
display(top5.round(2))

plt.figure(figsize=(14,4))
plt.bar(holdout["Week_Start_Date"].dt.strftime("%Y-%m-%d"), holdout["Error"])
plt.axhline(0, linewidth=1)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Error (Actual − Forecast)")
plt.title("Selected Model Holdout Errors")
plt.grid(axis="y", alpha=0.25)
plt.show()


## 9. Regression / demand drivers

Training-only multiple regression:
**Actual_Demand = β0 + β1 Price_Index + β2 Promo_Flag + β3 Event_Flag + β4 Economic_Index + error**

Interpret coefficients as associations holding the included variables constant. They are not automatically causal.


In [ ]:
X = train_df[["Price_Index","Promo_Flag","Event_Flag","Economic_Index"]].astype(float)
y = train_df["Actual_Demand"].astype(float)

X_sm = sm.add_constant(X)
reg = sm.OLS(y, X_sm).fit()
print(reg.summary())

reg_table = pd.DataFrame({
    "Coefficient": reg.params,
    "p-value": reg.pvalues
})
display(reg_table.round(4))
print("R-squared:", round(reg.rsquared, 3))
print("Adjusted R-squared:", round(reg.rsquared_adj, 3))


## 10. Stress testing the management decision


In [ ]:
base = 848.2
scenarios = pd.DataFrame({
    "Scenario": [
        "Downside -10%",
        "Base",
        "Upside +10%",
        "Event shock +5%",
        "15% availability loss"
    ],
    "12-week demand/serviceable units": [
        base*0.90,
        base,
        base*1.10,
        base*1.05,
        base*0.85
    ]
})
scenarios["Operational consequence"] = [
    "Lower production/procurement requirement; risk of excess if full upside capacity is committed.",
    "Firm baseline is covered; flexible option preserves responsiveness.",
    "Protected capacity of 935 is approximately sufficient for the upside case.",
    "Higher event-driven requirement; early promotion/event communication is important.",
    "Serviceable supply falls below base demand, creating an estimated shortage."
]
display(scenarios.round(1))


## 11. Forward 12-week planning forecast and recommendation

Management recommendation for the synthetic scenario:
- Expected 12-week demand: approximately **848 units**
- Firm commitment: **750 units**
- Flexible capacity/option: **185 units**
- Protected planning capacity: **935 units**
- Review trigger: reforecast if actual demand differs from plan by >10% for two consecutive weeks or if Tracking Signal shows sustained drift.


In [ ]:
# Refit the selected SES model on the full frozen history AFTER holdout evaluation.
# This is valid for the final forward plan because the historical holdout is now in the past.
full_y = df["Actual_Demand"].astype(float)
final_fit = SimpleExpSmoothing(
    full_y, initialization_method="estimated"
).fit(smoothing_level=0.6, optimized=False)

future_index = pd.date_range(
    df["Week_Start_Date"].iloc[-1] + pd.Timedelta(weeks=1),
    periods=12,
    freq="7D"
)
forward_fc = pd.Series(
    final_fit.forecast(12).to_numpy(),
    index=future_index,
    name="SES α=0.6 Forward Forecast"
)

display(forward_fc.to_frame().round(2))
print("Forward 12-week total:", round(forward_fc.sum(), 1))
print("Planning quantity used in the management recommendation: approximately 848 units.")


## 12. Audit checklist

- [x] Frozen synthetic dataset
- [x] 156 weekly observations
- [x] Final 12-week unseen holdout
- [x] Baseline + multiple competing methods
- [x] Error = Actual − Forecast
- [x] Required accuracy metrics
- [x] Five largest misses diagnosed
- [x] Training-only regression
- [x] No causal overclaim
- [x] Downside/upside/event/operational stress tests
- [x] Quantified recommendation and review trigger
- [x] Final forward forecast refitted on the full frozen history after holdout validation

**Important limitation:** the weekly Honda City 1.2L demand series and all derived model outputs are synthetic academic constructs, not Honda Atlas internal records.
